In [31]:
"""
level_valve_test.py
====================
Test script for the pneumatic level control valve connected to DAC0.
Ramps the valve fully open, holds, then ramps fully closed.

Hardware:
    LabJack T7  →  DAC0  →  I/P transducer  →  pneumatic valve

Wiring check before running:
    - Confirm DAC0 is wired to the valve signal input
    - Confirm instrument air supply is connected and at pressure
    - Confirm the valve is free to move (no manual override engaged)

Output voltage range:
    0.0 V  =  valve fully closed
    5.0 V  =  valve fully open
    (reverse these if your valve is air-to-close)
"""

from labjack import ljm
import time

# ── Configuration ──────────────────────────────────────────────────────────────
OUTPUT_REGISTER = "DAC0"
V_CLOSED        = 0.0     # Voltage corresponding to fully closed
V_OPEN          = 5.0     # Voltage corresponding to fully open
RAMP_STEPS      = 10      # Number of steps in each ramp
RAMP_DELAY_S    = 0.5     # Seconds between each ramp step
HOLD_TIME_S     = 120.0     # Seconds to hold at fully open before closing


def ramp(handle, v_start, v_end, steps, delay_s):
    """Ramp DAC0 linearly from v_start to v_end in `steps` increments."""
    for i in range(steps + 1):
        voltage = v_start + (v_end - v_start) * (i / steps)
        ljm.eWriteName(handle, OUTPUT_REGISTER, voltage)
        print(f"  DAC0 = {voltage:.2f} V")
        time.sleep(delay_s)


def main():
    handle = None
    try:
        # ── Connect ───────────────────────────────────────────────────────────
        handle = ljm.openS("T7", "ANY", "ANY")
        info   = ljm.getHandleInfo(handle)
        print(f"Connected to LabJack T7 [Serial: {info[2]}]")
        print(f"Output register: {OUTPUT_REGISTER}")
        print("-" * 40)

        # ── Safety: ensure valve starts closed ────────────────────────────────
        print("Initialising — setting valve CLOSED (0.0 V)...")
        ljm.eWriteName(handle, OUTPUT_REGISTER, V_CLOSED)
        time.sleep(2.0)
        
        # ── Open ──────────────────────────────────────────────────────────────
        print(f"\nRamping valve OPEN ({V_CLOSED} V → {V_OPEN} V)...")
        ramp(handle, V_CLOSED, V_OPEN, RAMP_STEPS, RAMP_DELAY_S)
        print(f"Valve fully open — holding for {HOLD_TIME_S} seconds...")
        time.sleep(HOLD_TIME_S)

        # ── Close ─────────────────────────────────────────────────────────────
        print(f"\nRamping valve CLOSED ({V_OPEN} V → {V_CLOSED} V)...")
        ramp(handle, V_OPEN, V_CLOSED, RAMP_STEPS, RAMP_DELAY_S)
        print("Valve fully closed.")

    except ljm.LJMError as e:
        print(f"\nLabJack error: {e}")

    except KeyboardInterrupt:
        print("\nTest interrupted by user.")
        
    finally:
        # ── Always close the valve and disconnect cleanly ─────────────────────
        if handle is not None:
            print("\nSafety close — setting DAC0 to 0.0 V before disconnecting...")
            ljm.eWriteName(handle, OUTPUT_REGISTER, 0.0)
            ljm.close(handle)
            print("LabJack disconnected.")


if __name__ == "__main__":
    main()

Connected to LabJack T7 [Serial: 470042305]
Output register: DAC0
----------------------------------------
Initialising — setting valve CLOSED (0.0 V)...

Ramping valve OPEN (0.0 V → 5.0 V)...
  DAC0 = 0.00 V
  DAC0 = 0.50 V
  DAC0 = 1.00 V
  DAC0 = 1.50 V
  DAC0 = 2.00 V
  DAC0 = 2.50 V
  DAC0 = 3.00 V
  DAC0 = 3.50 V
  DAC0 = 4.00 V
  DAC0 = 4.50 V
  DAC0 = 5.00 V
Valve fully open — holding for 120.0 seconds...

Ramping valve CLOSED (5.0 V → 0.0 V)...
  DAC0 = 5.00 V
  DAC0 = 4.50 V
  DAC0 = 4.00 V
  DAC0 = 3.50 V
  DAC0 = 3.00 V
  DAC0 = 2.50 V
  DAC0 = 2.00 V
  DAC0 = 1.50 V
  DAC0 = 1.00 V
  DAC0 = 0.50 V
  DAC0 = 0.00 V
Valve fully closed.

Safety close — setting DAC0 to 0.0 V before disconnecting...
LabJack disconnected.


In [30]:
"""
As above, but configured for DIO ports, using LJDAC. 
"""

from labjack import ljm
import time

# ── Configuration ──────────────────────────────────────────────────────────────
OUTPUT_REGISTER = "TDAC0"
V_CLOSED        = 0.0     # Voltage corresponding to fully closed
V_OPEN          = 5.0     # Voltage corresponding to fully open
RAMP_STEPS      = 10      # Number of steps in each ramp
RAMP_DELAY_S    = 0.5     # Seconds between each ramp step
HOLD_TIME_S     = 5.0     # Seconds to hold at fully open before closing


def ramp(handle, v_start, v_end, steps, delay_s):
    """Ramp DAC0 linearly from v_start to v_end in `steps` increments."""
    for i in range(steps + 1):
        voltage = v_start + (v_end - v_start) * (i / steps)
        ljm.eWriteName(handle, OUTPUT_REGISTER, voltage)
        print(f"  {OUTPUT_REGISTER} = {voltage:.2f} V")
        time.sleep(delay_s)


def main():
    handle = None
    try:
        # ── Connect ───────────────────────────────────────────────────────────
        handle = ljm.openS("T7", "ANY", "ANY")
        info   = ljm.getHandleInfo(handle)
        print(f"Connected to LabJack T7 [Serial: {info[2]}]")
        print(f"Output register: {OUTPUT_REGISTER}")
        print("-" * 40)

        # ── Safety: ensure valve starts closed ────────────────────────────────
        print("Initialising — setting valve CLOSED (0.0 V)...")
        ljm.eWriteName(handle, OUTPUT_REGISTER, V_CLOSED)
        time.sleep(2.0)
        
        # ── Open ──────────────────────────────────────────────────────────────
        print(f"\nRamping valve OPEN ({V_CLOSED} V → {V_OPEN} V)...")
        ramp(handle, V_CLOSED, V_OPEN, RAMP_STEPS, RAMP_DELAY_S)
        print(f"Valve fully open — holding for {HOLD_TIME_S} seconds...")
        time.sleep(HOLD_TIME_S)

        # ── Close ─────────────────────────────────────────────────────────────
        print(f"\nRamping valve CLOSED ({V_OPEN} V → {V_CLOSED} V)...")
        ramp(handle, V_OPEN, V_CLOSED, RAMP_STEPS, RAMP_DELAY_S)
        print("Valve fully closed.")

    except ljm.LJMError as e:
        print(f"\nLabJack error: {e}")

    except KeyboardInterrupt:
        print("\nTest interrupted by user.")
        
    finally:
        # ── Always close the valve and disconnect cleanly ─────────────────────
        if handle is not None:
            print(f"\nSafety close — setting {OUTPUT_REGISTER} to 0.0 V before disconnecting...")
            ljm.eWriteName(handle, OUTPUT_REGISTER, 0.0)
            ljm.close(handle)
            print("LabJack disconnected.")


if __name__ == "__main__":
    main()

Connected to LabJack T7 [Serial: 470042305]
Output register: TDAC0
----------------------------------------
Initialising — setting valve CLOSED (0.0 V)...

Ramping valve OPEN (0.0 V → 5.0 V)...
  TDAC0 = 0.00 V
  TDAC0 = 0.50 V
  TDAC0 = 1.00 V
  TDAC0 = 1.50 V
  TDAC0 = 2.00 V
  TDAC0 = 2.50 V
  TDAC0 = 3.00 V
  TDAC0 = 3.50 V
  TDAC0 = 4.00 V
  TDAC0 = 4.50 V
  TDAC0 = 5.00 V
Valve fully open — holding for 5.0 seconds...

Ramping valve CLOSED (5.0 V → 0.0 V)...
  TDAC0 = 5.00 V
  TDAC0 = 4.50 V
  TDAC0 = 4.00 V
  TDAC0 = 3.50 V
  TDAC0 = 3.00 V
  TDAC0 = 2.50 V
  TDAC0 = 2.00 V
  TDAC0 = 1.50 V
  TDAC0 = 1.00 V
  TDAC0 = 0.50 V
  TDAC0 = 0.00 V
Valve fully closed.

Safety close — setting TDAC0 to 0.0 V before disconnecting...
LabJack disconnected.
